# 03 · Base Model vs. Layout-Aware Document Pipeline

**Hardware**: 🟡 a self-hosted GLM-OCR server plus layout model, or the GLM-OCR MaaS API.

The checkpoint and the SDK are not the same experiment. This notebook keeps them separate and preserves the causal chain: page → boxes/labels → crops → per-region output → merge order → final Markdown. Use a non-sensitive test document with hosted APIs.

## 0. Setup

For a local pipeline install `glmocr[selfhosted]` and run the checkpoint behind vLLM or SGLang. For MaaS, plain `glmocr` plus `ZHIPU_API_KEY` is enough.

In [ ]:
# %pip install glmocr pillow requests
# %pip install "glmocr[selfhosted]"  # local PP-DocLayout-V3 path

import base64
import difflib
import io
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests
from PIL import Image, ImageDraw, ImageFont

BASE_URL = os.getenv("OCR_BASE_URL")
OUTPUT_DIR = Path("glm_ocr_artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Make a page whose layout we know

The oracle regions let us separate recogniser errors from detector errors. Replace the generated page with a real scan only after the ablation works.

In [ ]:
def font(size):
    for name in ("DejaVuSans.ttf", "Arial.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    return ImageFont.load_default()

page = Image.new("RGB", (1600, 1500), "white")
d = ImageDraw.Draw(page)
d.text((80, 60), "A Small Document Intelligence Test", fill="black", font=font(48))
left = ["LEFT COLUMN", "Batch ID: XQ-2049", "Pressure: 101.3 kPa", "Operator: Grace Hopper"]
right = ["RIGHT COLUMN", "Run date: 2026-08-17", "Temperature: 23.5 C", "Status: accepted"]
for i, line in enumerate(left): d.text((90, 210 + i * 58), line, fill="black", font=font(30))
for i, line in enumerate(right): d.text((850, 210 + i * 58), line, fill="black", font=font(30))

x, y = [100, 650, 1050, 1500], [620, 700, 780, 860]
for xx in x: d.line((xx, y[0], xx, y[-1]), fill="black", width=3)
for yy in y: d.line((x[0], yy, x[-1], yy), fill="black", width=3)
table = [("Sample", "Mass", "Score"), ("A-17", "12.40", "0.981"), ("B-03", "9.75", "0.947")]
for row_i, row in enumerate(table):
    for col_i, value in enumerate(row): d.text((x[col_i] + 16, y[row_i] + 17), value, fill="black", font=font(28))
d.text((100, 1040), "Energy model", fill="black", font=font(36))
d.text((100, 1110), "E = m c² + 1/2 k x²", fill="black", font=font(42))
d.text((100, 1300), "Conclusion: all validation checks passed.", fill="black", font=font(31))
PAGE_PATH = OUTPUT_DIR / "controlled_page.png"
page.save(PAGE_PATH)
page

In [ ]:
oracle_regions = [
    {"id": "title", "label": "text", "bbox": (60, 40, 1540, 145)},
    {"id": "left", "label": "text", "bbox": (60, 180, 790, 470)},
    {"id": "right", "label": "text", "bbox": (820, 180, 1550, 470)},
    {"id": "table", "label": "table", "bbox": (70, 590, 1530, 900)},
    {"id": "formula", "label": "formula", "bbox": (70, 1000, 1100, 1210)},
    {"id": "conclusion", "label": "text", "bbox": (70, 1260, 1500, 1380)},
]
PROMPTS = {"text": "Text Recognition:", "table": "Table Recognition:", "formula": "Formula Recognition:"}
preview = page.copy()
pd = ImageDraw.Draw(preview)
for region in oracle_regions:
    pd.rectangle(region["bbox"], outline="red", width=4)
    pd.text(region["bbox"][:2], f"{region['id']}:{region['label']}", fill="red", font=font(20))
preview

## 2. Call only the base-model server

This endpoint does not run PP-DocLayout-V3 or reconstruct reading order.

In [ ]:
def data_uri(image):
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()

def base_ocr(image, prompt, url=BASE_URL):
    if not url:
        raise RuntimeError("Set OCR_BASE_URL to the checkpoint's OpenAI-compatible endpoint")
    payload = {
        "model": "glm-ocr",
        "messages": [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": data_uri(image)}},
            {"type": "text", "text": prompt},
        ]}],
        "temperature": 0,
        "max_tokens": 4096,
    }
    started = time.perf_counter()
    response = requests.post(url, json=payload, timeout=300)
    response.raise_for_status()
    value = response.json()
    return {"text": value["choices"][0]["message"]["content"], "seconds": time.perf_counter() - started}

if BASE_URL:
    whole_page = base_ocr(page, "Text Recognition:")
    print(whole_page["text"])
else:
    whole_page = None
    print("Set OCR_BASE_URL to run base-model conditions.")

## 3. Oracle crops isolate recognition from detection

The detector cannot be blamed in this condition. Serial and parallel calls should preserve content while changing wall time.

In [ ]:
def recognise_region(region):
    result = base_ocr(page.crop(region["bbox"]), PROMPTS[region["label"]])
    return {**region, **result}

if BASE_URL:
    started = time.perf_counter()
    serial_regions = [recognise_region(r) for r in oracle_regions]
    serial_seconds = time.perf_counter() - started
    started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=6) as pool:
        parallel_regions = list(pool.map(recognise_region, oracle_regions))
    parallel_seconds = time.perf_counter() - started
    print(f"serial={serial_seconds:.2f}s parallel={parallel_seconds:.2f}s")
    for region in parallel_regions:
        print(f"\n[{region['id']} / {region['label']}]\n{region['text']}")
else:
    serial_regions = parallel_regions = []

## 4. Inject upstream failures

Omit the formula box and mislabel the table as text. The first makes recovery impossible; the second preserves pixels but requests the wrong output language.

In [ ]:
missed_formula = [r for r in oracle_regions if r["id"] != "formula"]
wrong_table = [{**r, "label": "text"} if r["id"] == "table" else r for r in oracle_regions]
if BASE_URL:
    omitted_outputs = [recognise_region(r) for r in missed_formula]
    wrong_label_output = recognise_region(next(r for r in wrong_table if r["id"] == "table"))
    print("table with Text Recognition prompt:\n", wrong_label_output["text"])
    assert all(r["id"] != "formula" for r in omitted_outputs)

## 5. Run the complete SDK pipeline

The SDK call includes page loading, layout detection, region OCR, and formatting. In MaaS mode the service performs all four; in self-hosted mode the SDK uses your layout model and OCR endpoint.

In [ ]:
RUN_SDK = os.getenv("RUN_GLMOCR_SDK", "0") == "1"
if RUN_SDK:
    from glmocr import GlmOcr
    mode = os.getenv("GLMOCR_MODE", "maas")
    kwargs = {"mode": mode}
    if os.getenv("ZHIPU_API_KEY"):
        kwargs["api_key"] = os.environ["ZHIPU_API_KEY"]
    with GlmOcr(**kwargs) as parser:
        pipeline_result = parser.parse(str(PAGE_PATH))
        pipeline_result.save(output_dir=str(OUTPUT_DIR / "sdk_result"))
    print(pipeline_result.markdown_result)
    print("layout regions:", len(pipeline_result.layout_details or []))
else:
    pipeline_result = None
    print("Set RUN_GLMOCR_SDK=1 after configuring MaaS or self-hosted mode.")

## 6. Cheap metrics that catch expensive mistakes

Whole-text similarity and digit precision/recall answer different questions. Keep both.

In [ ]:
REFERENCE_LINES = [
    "A Small Document Intelligence Test", "LEFT COLUMN", "Batch ID: XQ-2049",
    "Pressure: 101.3 kPa", "Operator: Grace Hopper", "RIGHT COLUMN",
    "Run date: 2026-08-17", "Temperature: 23.5 C", "Status: accepted",
    "Sample Mass Score", "A-17 12.40 0.981", "B-03 9.75 0.947",
    "Energy model", "E = m c² + 1/2 k x²", "Conclusion: all validation checks passed.",
]
REFERENCE = "\n".join(REFERENCE_LINES)

def normalise(text):
    return re.sub(r"\s+", " ", text).strip().lower()

def text_similarity(reference, prediction):
    return difflib.SequenceMatcher(None, normalise(reference), normalise(prediction)).ratio()

def digit_prf(reference, prediction):
    pattern = r"[-+]?\d+(?:[.,]\d+)*"
    ref, pred = Counter(re.findall(pattern, reference)), Counter(re.findall(pattern, prediction))
    correct = sum((ref & pred).values())
    return {
        "precision": correct / sum(pred.values()) if pred else 0.0,
        "recall": correct / sum(ref.values()) if ref else 0.0,
        "missing": list((ref - pred).elements()),
        "invented": list((pred - ref).elements()),
    }

conditions = {}
if whole_page: conditions["whole_page_base"] = whole_page["text"]
if parallel_regions: conditions["oracle_crops"] = "\n".join(r["text"] for r in parallel_regions)
if pipeline_result: conditions["sdk_pipeline"] = pipeline_result.markdown_result
for name, prediction in conditions.items():
    print(name, {"text_similarity": text_similarity(REFERENCE, prediction), "digits": digit_prf(REFERENCE, prediction)})

## What to save for every failed page

1. rendered page and DPI;
2. detector boxes/polygons, labels, and confidence;
3. exact crop images;
4. prompt and raw output for every crop;
5. merge order and post-processing operations;
6. final Markdown/JSON and per-contract metrics.

If a pipeline cannot produce this bundle, it is not yet debuggable.

## Exercises

1. Swap the two column boxes in merge order while keeping recognition perfect. Which metric catches it?
2. Shrink the table crop until the outer rules disappear. Does the model lose text, structure, or both?
3. Compare bbox crops with polygon masks on a rotated receipt. Score detection and recognition separately.
4. Add retries for HTTP 429/503 and ensure retries cannot duplicate regions.
5. Run the same pages through chapter 24's DeepSeek-OCR-2 notebook and compare by slice, not only by mean.